# Lab 7 — RLHF & DPO Alignment

Pretraining teaches a model *capabilities* — the ability to produce grammatical text that's loosely correlated with the prompt. **Alignment** teaches a model *what kind of response is desired* — helpful, honest, harmless, concise, faithful to your company's policy, whatever you define.

Every frontier model you've heard of — GPT-4, Claude, Gemini, Llama 3 — went through an alignment pipeline. Before alignment, a raw LLM might respond to *"What's the capital of France?"* with *"What's the capital of France? What's the capital of Germany? What's the capital of..."* (continuing the pattern), or with a 500-word essay, or with an irrelevant tangent. After alignment, it says *"Paris."*

### The RLHF recipe (OpenAI, 2017–2022)

The method that made ChatGPT possible, introduced across [Christiano et al. 2017](https://arxiv.org/abs/1706.03741), [Ziegler et al. 2019](https://arxiv.org/abs/1909.08593), and [OpenAI's InstructGPT paper (Ouyang et al., 2022)](https://arxiv.org/abs/2203.02155). It has **three stages**:

1. **Supervised fine-tuning (SFT)** — train the base model on high-quality demonstrations
2. **Reward model training** — humans rank pairs of outputs; train a separate model to predict the ranking
3. **RL (PPO) fine-tuning** — use the reward model to fine-tune the SFT model via reinforcement learning, with a KL penalty to prevent drift

It works, but it is *famously* fiddly. You're juggling three models (base, SFT, reward, policy), unstable RL, reward hacking, and a full PPO implementation. Reproducing InstructGPT-scale RLHF was an extreme engineering effort.

### DPO: the 2023 simplification

**Rafailov et al., 2023 — [Direct Preference Optimization: Your Language Model Is Secretly a Reward Model](https://arxiv.org/abs/2305.18290).** The insight hiding in the title: the *closed-form* mapping between reward functions and optimal policies lets you skip the reward model entirely. You can train directly on preference pairs (chosen vs rejected) with a simple classification-style loss.

DPO replaces the full RLHF stack with **one training loop, one model being updated, one frozen reference model, one contrastive loss**. Published results match or beat PPO-RLHF on summarization and single-turn dialogue. It's now the default alignment algorithm in open-source — Zephyr, Tulu, Orca, Mistral's Instruct variants, Qwen's chat variants all use DPO or close derivatives (IPO, KTO, SimPO).

### What you'll do in this lab

1. **Load GPT-2** (base model) and generate baseline outputs on a test set of prompts.
2. **Build a preference dataset** — 10 prompts with (chosen, rejected) response pairs encoding a concrete alignment preference.
3. **Run DPO** with `trl.DPOTrainer` on the preference data — a few hundred steps, two minutes on a 3060 Ti.
4. **Regenerate on the same prompts** with the aligned model and observe the shift.

We'll use GPT-2 because it's small (124M params, <1s per step), and the alignment *signal* is visible even with this tiny preference set. The pipeline you build here scales unchanged to Llama-70B.

### Read while this runs

- **[DPO paper (Rafailov et al., 2023)](https://arxiv.org/abs/2305.18290)** — sections 4 and 5 derive the loss; the math is surprisingly clean.
- **[Tyler Romero — DPO Explained In-Depth](https://www.tylerromero.com/posts/2024-04-dpo/)** — best single blog post covering the intuition and the gradient.
- **[TRL docs — DPOTrainer](https://huggingface.co/docs/trl/main/dpo_trainer)** — what we're driving here.

---

## Step 1 — Load the base model and capture baseline outputs

We pick three prompts that expose classic base-LM failure modes:

- Completion: a model trained on raw text often just **continues** a question rather than answering it
- Conciseness: base LMs ramble; aligned ones stay terse
- Directness: base LMs over-hedge

After DPO, if our preference dataset rewards concise direct answers, we should see movement on all three.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'gpt2'

base_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to('cuda')
base_model.eval()
print(f'Loaded {MODEL_ID}: {base_model.num_parameters()/1e6:.0f}M params')

# Keep a frozen copy of the base model — DPO needs the reference model to stay put
# (we'll use this directly via TRL's ref_model=None default, which auto-copies)

test_prompts = [
    'Q: What is the capital of France?\nA:',
    'Q: How do I boil an egg?\nA:',
    'Q: Explain gravity in one sentence.\nA:',
]

def generate(model, tok, prompt, max_new=25):
    ids = tok(prompt, return_tensors='pt').input_ids.to('cuda')
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

baseline_outputs = {p: generate(base_model, base_tokenizer, p) for p in test_prompts}
print('\nBASELINE OUTPUTS (before alignment):')
for p, t in baseline_outputs.items():
    print(f'\n  PROMPT: {p.strip()}')
    print(f'  BASE:   {t!r}')

/usr/local/lib/python3.11/dist-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Loaded gpt2: 124M params

BASELINE OUTPUTS (before alignment):

  PROMPT: Q: What is the capital of France?
A:
  BASE:   'The capital of France is Paris.\nQ: What is the capital of France?\nA: The capital of France is'

  PROMPT: Q: How do I boil an egg?
A:
  BASE:   'I boil an egg.\nQ: How do I boil an egg?\nA: I boil an egg.\nQ'

  PROMPT: Q: Explain gravity in one sentence.
A:
  BASE:   "It's a very simple thing. It's a simple thing. It's a simple thing. It's a simple thing."


In [2]:
from preporato_labs import Lab
lab = Lab('dpo-alignment')
lab.check(1)

OK — base model on cuda:0, 3 baseline generations captured
STEP_PASSED


Step 1 Complete! Scroll down to continue...

True

## Step 2 — Build the preference dataset

DPO's input is a set of `(prompt, chosen, rejected)` triples. The *same* prompt gets paired with two responses — one we want the model to produce more often, one less. The model learns the relative preference.

Real preference datasets (Anthropic HH-RLHF, UltraFeedback, Nectar) have **10K–1M+ pairs** collected from crowdworkers or other LLMs. We'll hand-craft 10. The signal is weak but directional — after ~50 steps you should see the aligned model trending toward the chosen style.

Our preference: **terse, direct, factual answers**. Rejected: rambling, uncertain, pattern-continuing.

### 🐛 Common mistake: chosen == rejected (or nearly so)

If your chosen and rejected responses differ only in tiny stylistic ways, the preference signal is too weak to learn from. Make them *noticeably* different on the axis you care about. If you care about conciseness, chosen should be *much* shorter. If you care about factuality, chosen should be *right* and rejected *wrong*. Weak contrast → flat loss → no learning.

In [3]:
from datasets import Dataset

# Small hand-crafted preference set. In production you'd use UltraFeedback or similar (~60k pairs).
pairs = [
    {'prompt': 'Q: What is the capital of France?\nA:',
     'chosen': ' Paris.',
     'rejected': ' What is the capital of Germany? What is the capital of Italy? What is'},
    {'prompt': 'Q: Define machine learning in one sentence.\nA:',
     'chosen': ' Machine learning is the use of statistical models that improve on a task from data rather than explicit rules.',
     'rejected': ' Machine learning, which is also sometimes called artificial intelligence or AI, is, broadly speaking, a'},
    {'prompt': 'Q: How do I boil an egg?\nA:',
     'chosen': ' Place eggs in cold water, bring to a boil, then simmer for 6-10 minutes depending on doneness.',
     'rejected': ' Well, the question of how to boil an egg is one that has been asked by many'},
    {'prompt': 'Q: What is 7 times 8?\nA:',
     'chosen': ' 56.',
     'rejected': ' 7 times 8 is a multiplication problem. Multiplication is one of the four basic operations'},
    {'prompt': 'Q: Explain gravity in one sentence.\nA:',
     'chosen': ' Gravity is the attractive force between objects with mass.',
     'rejected': ' Gravity. Gravity is a force. Gravity is an invisible force. Gravity is a force that'},
    {'prompt': 'Q: When did World War II end?\nA:',
     'chosen': ' 1945.',
     'rejected': ' World War II, often abbreviated as WW2 or WWII, ended at some point in the'},
    {'prompt': 'Q: What does DNA stand for?\nA:',
     'chosen': ' Deoxyribonucleic acid.',
     'rejected': ' DNA. DNA stands for a long word. DNA is in every cell. DNA is made'},
    {'prompt': 'Q: Who wrote Hamlet?\nA:',
     'chosen': ' William Shakespeare.',
     'rejected': ' Hamlet is a play. Who wrote Hamlet? Who wrote Macbeth? Who wrote Othello'},
    {'prompt': 'Q: What is photosynthesis?\nA:',
     'chosen': ' Photosynthesis is the process by which plants convert sunlight into chemical energy stored as glucose.',
     'rejected': ' Photosynthesis is a word. Photosynthesis is a process. Photosynthesis happens in plants. Photosynthesis'},
    {'prompt': 'Q: What is the speed of light?\nA:',
     'chosen': ' Approximately 300,000 kilometers per second in a vacuum.',
     'rejected': ' The speed of light is very fast. The speed of light is a number. The speed of'},
]

preference_dataset = Dataset.from_list(pairs)
print(f'Preference dataset: {len(preference_dataset)} pairs with columns {preference_dataset.column_names}')
print(f'\nExample pair:')
row = preference_dataset[0]
print(f"  PROMPT:   {row['prompt']!r}")
print(f"  CHOSEN:   {row['chosen']!r}")
print(f"  REJECTED: {row['rejected']!r}")

Preference dataset: 10 pairs with columns ['prompt', 'chosen', 'rejected']

Example pair:
  PROMPT:   'Q: What is the capital of France?\nA:'
  CHOSEN:   ' Paris.'
  REJECTED: ' What is the capital of Germany? What is the capital of Italy? What is'


In [4]:
lab.check(2)

OK — preference_dataset: 10 pairs with correct {'chosen', 'prompt', 'rejected'} schema
STEP_PASSED


Step 2 Complete! Scroll down to continue...

True

## Step 3 — Train with DPOTrainer

### The DPO loss in one line

For each (prompt, chosen, rejected) triple, the loss is:

```
L_DPO = -log σ( β · [log π(chosen) - log π_ref(chosen)] - β · [log π(rejected) - log π_ref(rejected)] )
```

where `π` is the policy (our model we're training), `π_ref` is the frozen reference (the base model at t=0), and `β` is a KL-divergence strength coefficient (typically 0.1–0.5).

Read it as: *maximize the log-ratio of chosen-prob to rejected-prob, regularized by how far we've drifted from the reference.* The loss **decreases** when the policy puts MORE probability mass on chosen and LESS on rejected, relative to the reference.

### Why a reference model?

Without the `π_ref` anchor, the model can shift all its probability mass to any chosen sequence (even nonsensical ones) and call it a day — collapsing fluency. The KL penalty via the reference keeps it *recognizable* as the base model's distribution, just tilted toward preferred responses.

TRL's `DPOTrainer` handles this automatically. Pass `ref_model=None` and it clones the model at init time.

In [5]:
from trl import DPOTrainer, DPOConfig

# Small but real training run. On a 3060 Ti GPT-2 fits with room to spare.
config = DPOConfig(
    output_dir='/tmp/dpo_gpt2',
    per_device_train_batch_size=2,
    num_train_epochs=10,                 # 10 * 10 pairs = ~100 gradient steps
    learning_rate=5e-6,
    logging_steps=10,
    beta=0.1,                            # KL strength — lower = more freedom to diverge
    max_length=128,
    save_strategy='no',
    report_to='none',
)

trainer = DPOTrainer(
    model=base_model,                    # the policy we update
    ref_model=None,                      # TRL will clone base_model as the frozen reference
    args=config,
    train_dataset=preference_dataset,
    processing_class=base_tokenizer,
)

print(f'Training DPO on {len(preference_dataset)} preference pairs, {config.num_train_epochs} epochs...')
train_result = trainer.train()
print(f'\nFinal train_loss: {train_result.metrics["train_loss"]:.4f}')

aligned_model = trainer.model
aligned_model.eval()


Adding EOS to train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Training DPO on 10 preference pairs, 10 epochs...


Step,Training Loss
10,0.469400
20,0.169000
30,0.080400
40,0.050300
50,0.038400



Final train_loss: 0.1615


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0, inplace=False)
          (resid_dropout): Dropout(p=0, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [7]:
lab.check(3)

OK — DPO trained 50 steps, final loss=0.1615 (behavior shift verified by the baseline-vs-aligned generations in the next step)
STEP_PASSED


Step 3 Complete! Scroll down to continue...

True

## Step 4 — Before vs after: did alignment work?

The moment of truth. We regenerate on the **exact same three prompts** we used in Step 1, using the DPO-trained model, and compare side by side.

What you're looking for:
- Shorter outputs (our preferred style was terse)
- Actual answers instead of pattern continuations
- More direct wording

Caveat: 10 preference pairs and 100 steps on a 124M-parameter model is a *tiny* amount of alignment signal. On a production-scale run you'd use 50K+ pairs and a 7B+ model and you'd see dramatic quality shifts. Here, modest directional shifts are the realistic expectation.

**Your homework when inspecting the outputs:** did they shift in the right *direction*? Don't expect them to become correct answers — GPT-2 doesn't know who wrote Hamlet. Expect them to become **more confident, terser, and less pattern-repetitive.**

In [8]:
aligned_outputs = {p: generate(aligned_model, base_tokenizer, p) for p in test_prompts}

print('=' * 90)
print('BEFORE vs AFTER DPO ALIGNMENT')
print('=' * 90)
for p in test_prompts:
    print(f'\n PROMPT: {p.strip()}')
    print(f'  BEFORE: {baseline_outputs[p]!r}')
    print(f'  AFTER : {aligned_outputs[p]!r}')
    delta_len = len(aligned_outputs[p]) - len(baseline_outputs[p])
    print(f'  length delta: {delta_len:+d} chars')

BEFORE vs AFTER DPO ALIGNMENT

 PROMPT: Q: What is the capital of France?
A:
  BEFORE: 'The capital of France is Paris.\nQ: What is the capital of France?\nA: The capital of France is'
  AFTER : 'Paris.\nQ: What is the capital of France?\nA: Paris.\nQ: What is the capital of'
  length delta: -17 chars

 PROMPT: Q: How do I boil an egg?
A:
  BEFORE: 'I boil an egg.\nQ: How do I boil an egg?\nA: I boil an egg.\nQ'
  AFTER : 'In a small saucepan, combine the eggs, water, and salt. Add the chicken and cook for about 5 minutes,'
  length delta: +42 chars

 PROMPT: Q: Explain gravity in one sentence.
A:
  BEFORE: "It's a very simple thing. It's a simple thing. It's a simple thing. It's a simple thing."
  AFTER : "It's not that simple. It's that simple because the equations of gravity are not the same as the equations of mass."
  length delta: +26 chars


In [9]:
lab.check(4)

OK — aligned model produced different outputs vs baseline on 3/3 prompts (alignment signal is measurable)
STEP_PASSED


Step 4 Complete! Lab complete!

True

---

## What you just built

An end-to-end DPO alignment pipeline on GPT-2 — **the same pipeline that Zephyr, Tulu, and the post-2023 generation of open chat models use**, just at smaller scale. The three shifts from RLHF that DPO delivers:

1. **One model update loop instead of three** (no reward model, no PPO)
2. **Standard supervised-learning optimizers** (AdamW) instead of RL machinery
3. **A single implicit KL anchor** via the reference model, not explicit reward shaping

## What to read next

- **[DPO paper (Rafailov et al., 2023)](https://arxiv.org/abs/2305.18290)** — the derivation.
- **[Zephyr-7B paper (HuggingFace H4 team, 2023)](https://arxiv.org/abs/2310.16944)** — the first major open model to use DPO at scale. Ablations worth studying.
- **[IPO — Identity Preference Optimization (Azar et al., 2023)](https://arxiv.org/abs/2310.12036)** — fixes a subtle DPO overfitting issue on noisy preferences.
- **[KTO — Kahneman-Tversky Optimization (Ethayarajh et al., 2024)](https://arxiv.org/abs/2402.01306)** — handles prospect-theory effects in preferences; doesn't require pairs.
- **[SimPO (Meng et al., 2024)](https://arxiv.org/abs/2405.14734)** — removes the reference model entirely for memory savings.
- **[Constitutional AI (Bai et al., 2022)](https://arxiv.org/abs/2212.08073)** — Anthropic's alternative: use an LLM to generate the preferences.

## What to try next

- Replace hand-crafted pairs with the [UltraFeedback](https://huggingface.co/datasets/openbmb/UltraFeedback) dataset (~60K pairs) and re-train — you'll need a larger model for the extra signal to matter.
- Swap `DPOTrainer` for `KTOTrainer` (also in TRL) and compare convergence curves.
- Sweep `beta` across `[0.01, 0.1, 0.5, 1.0]` — observe the tradeoff between alignment strength and fluency loss.